In [3]:
#############################################################
# GABUNGKAN FILE HOMOGENISASI PER PARAMETER & BASELINE (DENGAN DETEKSI KOLOM DINAMIS)
#############################################################
import pandas as pd
import glob
import os
import re
import shutil
from pathlib import Path

# ==============================
# 1. KONFIGURASI
# ==============================
DATA_DIR = 'data' 
HOMO_DIR   = os.path.join(DATA_DIR, '03.QC_Dataset_Level_02_Homogenisasi')
OUTPUT_DIR = os.path.join(DATA_DIR, '04.Dataset_Final')

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Parameter dan baseline
PARAMS = [
    'TEMPERATURE_AVG_C',
    'TEMP_24H_TN_C',
    'TEMP_24H_TX_C',
]
BASELINES = ['1981', '1991']

# 🔑 MAPPING PARAMETER KE KOLOM NILAI (CRITICAL FIX)
PARAM_TO_VALUE_COL = {
    'TEMPERATURE_AVG_C': [f'HOMO_TEMPERATURE_AVG_C','temperature', 'temp_avg', 'avg_temp', 'temp', 'value', 'VALUE'],
    'TEMP_24H_TN_C':     ['HOMO_TEMP_24H_TN_C', 'min_temp', 'temperature_min', 'temp_min', 'value', 'VALUE'],
    'TEMP_24H_TX_C':     ['HOMO_TEMP_24H_TX_C', 'max_temp', 'temperature_max', 'temp_max', 'value', 'VALUE'],
}

PARAM_TO_TIMESTAMP_COL = ['DATA_TIMESTAMP', 'timestamp', 'time', 'date', 'datetime', 'observation_time']

# ==============================
# 2. VALIDASI KRUSIAL: AMBIL DAFTAR WMO_ID VALID DARI SUMMARY (DINAMIS)
# ==============================
def get_valid_wmo_ids(param, baseline):
    """
    Ambil WMO_ID yang memenuhi kriteria 80% availability.
    Mendukung struktur kolom yang berbeda antar parameter.
    """
    # Mapping parameter ke direktori summary (semua parameter TN/TX menggunakan summary TEMPERATURE_AVG_C)
    summary_path = os.path.join(
        DATA_DIR, 
        '01.QC_Dataset_Level_01', 
        param, 
        '05.Summary', 
        '00.Summary_80percent.csv'
    )
    
    if not os.path.exists(summary_path):
        raise FileNotFoundError(f"File summary tidak ditemukan: {summary_path}")
    
    avail = pd.read_csv(summary_path)
    
    # 🔑 Tentukan kolom ketersediaan dengan strategi fallback dinamis
    col_candidates = [
        f"80%_{param}_{baseline}",
        f"80%_{param}_1981",
        f"80%_{param}_{baseline}",
        f"80%_{param}_1981",
    ]
    
    col_name = None
    for candidate in col_candidates:
        if candidate in avail.columns:
            col_name = candidate
            break
    
    if col_name is None:
        # Cari kolom yang mengandung pola '80%' dan 'TEMP'
        temp_cols = [c for c in avail.columns if '80%' in c and ('TEMP' in c or 'temperature' in c.lower())]
        if temp_cols:
            col_name = temp_cols[0]
        else:
            raise ValueError(
                f"Tidak dapat menemukan kolom ketersediaan 80%. "
                f"Kolom tersedia: {list(avail.columns)}"
            )
    
    # 🔑 Filter stasiun valid
    valid_mask = avail[col_name] == True
    valid_wmos = set(avail[valid_mask]['WMO_ID'].astype(str).unique())
    
    return valid_wmos

# ==============================
# 3. DETEKSI KOLOM NILAI & TIMESTAMP SECARA DINAMIS
# ==============================
def detect_columns(file_path, param):
    """
    Deteksi kolom nilai dan timestamp yang sesuai dari file CSV.
    Mengembalikan: (value_col, timestamp_col, all_columns)
    """
    # Baca header saja
    df_head = pd.read_csv(file_path, nrows=0)
    cols = [c.lower() for c in df_head.columns]
    original_cols = list(df_head.columns)
    
    # 🔑 Deteksi kolom nilai berdasarkan mapping parameter
    value_candidates = PARAM_TO_VALUE_COL.get(param, ['value', 'VALUE'])
    value_col = None
    
    for candidate in value_candidates:
        if candidate.lower() in cols:
            # Ambil nama kolom original (case-sensitive)
            idx = cols.index(candidate.lower())
            value_col = original_cols[idx]
            break
    
    # 🔑 Deteksi kolom timestamp
    timestamp_col = None
    for candidate in PARAM_TO_TIMESTAMP_COL:
        if candidate.lower() in cols:
            idx = cols.index(candidate.lower())
            timestamp_col = original_cols[idx]
            break
    
    return value_col, timestamp_col, original_cols

# ==============================
# 4. PENGECEKAN INTEGRITAS FILE DENGAN DETEKSI KOLOM
# ==============================
def check_file_integrity(file_path, valid_wmos, param):
    """Validasi komprehensif dengan deteksi kolom dinamis"""
    issues = []
    filename = os.path.basename(file_path)
    
    # 🔒 CEK 1: Pola nama file
    if not re.match(r'^WMO_\d{5,6}_homogen_(fallback|daily)\.csv$', filename):
        issues.append("Pola nama tidak sesuai standar")
    
    # 🔒 CEK 2: Ekstrak WMO_ID dari nama file
    wmo_match = re.search(r'WMO_(\d{5,6})_', filename)
    if not wmo_match:
        issues.append("Tidak dapat ekstrak WMO_ID dari nama file")
        return filename, None, issues, "NAMA_FILE_INVALID"
    
    wmo_id = wmo_match.group(1)
    
    # 🔒 CEK 3: WMO_ID harus ada di daftar valid
    if wmo_id not in valid_wmos:
        issues.append(f"Tidak memenuhi kriteria 80% availability")
        return filename, wmo_id, issues, "TIDAK_MEMENUHI_80PCT"
    
    # 🔒 CEK 4: File harus ada dan tidak kosong
    if not os.path.exists(file_path):
        issues.append("File tidak ditemukan")
        return filename, wmo_id, issues, "FILE_HILANG"
    
    if os.path.getsize(file_path) < 100:
        issues.append("File terlalu kecil (<100 bytes)")
        return filename, wmo_id, issues, "FILE_KOSONG"
    
    # 🔒 CEK 5: Deteksi kolom kritis (nilai & timestamp)
    try:
        value_col, timestamp_col, all_cols = detect_columns(file_path, param)
        
        if value_col is None:
            issues.append(f"Kolom nilai tidak ditemukan. Kolom tersedia: {all_cols}")
            return filename, wmo_id, issues, "KOLOM_NILAI_HILANG"
        
        if timestamp_col is None:
            issues.append(f"Kolom timestamp tidak ditemukan. Kolom tersedia: {all_cols}")
            return filename, wmo_id, issues, "KOLOM_TIMESTAMP_HILANG"
        
        # Simpan info kolom untuk pemrosesan nanti
        return filename, wmo_id, issues, "VALID", value_col, timestamp_col
        
    except Exception as e:
        issues.append(f"Gagal deteksi kolom: {str(e)}")
        return filename, wmo_id, issues, "GAGAL_DETEKSI_KOLOM", None, None
    
    return filename, wmo_id, issues, "VALID", None, None

# ==============================
# 5. FUNGSI GABUNG DENGAN DETEKSI KOLOM DINAMIS
# ==============================
def gabung_homogenisasi(param, baseline):
    print(f"\n▶ Menggabungkan hasil homogenisasi: {param} | Baseline: {baseline}")
    
    in_dir  = os.path.join(HOMO_DIR, f"{param}_BASELINE_{baseline}", 'data')
    out_csv = os.path.join(OUTPUT_DIR, f'{param}_homogen_final_baseline_{baseline}.csv')
    
    # 🔒 VALIDASI 0: Pastikan direktori input ada
    if not os.path.isdir(in_dir):
        print(f"  ❌ Direktori input tidak ditemukan: {in_dir}")
        return False
    
    # 🔒 VALIDASI 1: Ambil daftar WMO_ID valid dari summary
    try:
        valid_wmos = get_valid_wmo_ids(param, baseline)
        print(f"  ✅ Terdapat {len(valid_wmos)} WMO_ID valid (≥80% availability)")
    except Exception as e:
        print(f"  ❌ Gagal validasi WMO_ID: {e}")
        import traceback
        traceback.print_exc()
        return False
    
    # 🔒 VALIDASI 2: Temukan file dengan pola ketat
    pattern = os.path.join(in_dir, "WMO_[0-9]*_homogen_*.csv")
    all_files = sorted(glob.glob(pattern))
    
    if not all_files:
        print(f"  ⚠️ Tidak ada file ditemukan dengan pola: {pattern}")
        return False
    
    print(f"  Menemukan {len(all_files)} file kandidat. Memulai validasi integritas...")
    
    # 🔒 VALIDASI 3: Filter file dengan pengecekan integritas + deteksi kolom
    valid_files = []  # [(file_path, wmo_id, value_col, timestamp_col)]
    rejected_files = []  # [(filename, wmo_id, reason, category)]
    
    for f in all_files:
        result = check_file_integrity(f, valid_wmos, param)
        filename, wmo_id, issues, category = result[0], result[1], result[2], result[3]
        
        if category == "VALID" and not issues:
            # Ekstrak kolom dari hasil deteksi
            value_col, timestamp_col = result[4], result[5]
            valid_files.append((f, wmo_id, value_col, timestamp_col))
        else:
            rejected_files.append((filename, wmo_id, "; ".join(issues), category))
    
    # 🔒 LAPORAN VALIDASI
    print(f"\n  ✅ File VALID   : {len(valid_files)}")
    print(f"  ❌ File DITOLAK : {len(rejected_files)}")
    
    if rejected_files:
        print(f"\n  Detail penolakan (kategori):")
        from collections import Counter
        cat_count = Counter([cat for _,_,_,cat in rejected_files])
        for cat, cnt in cat_count.most_common():
            print(f"    - {cat}: {cnt} file")
        
        # Tampilkan contoh penolakan
        if rejected_files:
            print(f"\n  Contoh file ditolak:")
            for i, (fname, wmo_id, reason, cat) in enumerate(rejected_files[:3], 1):
                wmo_display = wmo_id if wmo_id else "N/A"
                print(f"    {i}. [{cat}] WMO_{wmo_display}: {fname}")
                print(f"        Alasan: {reason[:100]}...")
    
    # 🔒 VALIDASI 4: Pastikan ada file valid untuk diproses
    if not valid_files:
        # 🔑 DIAGNOSTIK KRUSIAL: Periksa struktur kolom 1 file sampel
        if all_files:
            sample_file = all_files[0]
            print(f"\n  🔍 DIAGNOSTIK: Memeriksa struktur kolom file sampel: {os.path.basename(sample_file)}")
            try:
                sample_df = pd.read_csv(sample_file, nrows=0)
                print(f"     Kolom tersedia: {list(sample_df.columns)}")
                print(f"     Mapping yang diharapkan untuk {param}: {PARAM_TO_VALUE_COL.get(param, [])}")
            except Exception as e:
                print(f"     Gagal baca header: {e}")
        
        print(f"\n  ⚠️ Tidak ada file valid untuk diproses!")
        return False
    
    # Proses hanya file valid
    print(f"\n  Memproses {len(valid_files)} file valid...")
    dfs = []
    processing_errors = []
    
    for f_path, wmo_id, value_col, timestamp_col in valid_files:
        try:
            # 🔑 Baca dengan kolom timestamp yang terdeteksi
            parse_dates = [timestamp_col] if timestamp_col else None
            df = pd.read_csv(f_path, parse_dates=parse_dates, low_memory=False)
            
            # 🔑 Normalisasi struktur kolom ke format standar
            # Rename kolom nilai ke 'VALUE' untuk konsistensi
            if value_col != 'VALUE':
                df = df.rename(columns={value_col: 'VALUE'})
            
            # Rename kolom timestamp ke 'DATA_TIMESTAMP' untuk konsistensi
            if timestamp_col and timestamp_col != 'DATA_TIMESTAMP':
                df = df.rename(columns={timestamp_col: 'DATA_TIMESTAMP'})
            
            # Pastikan kolom wajib ada
            required_cols = ['WMO_ID', 'DATA_TIMESTAMP', 'VALUE']
            missing = [c for c in required_cols if c not in df.columns]
            if missing:
                raise ValueError(f"Kolom wajib hilang setelah renaming: {missing}. Kolom tersedia: {list(df.columns)}")
            
            # Normalisasi tipe data
            df['WMO_ID'] = df['WMO_ID'].astype(str)
            df['VALUE'] = pd.to_numeric(df['VALUE'], errors='coerce')
            
            # Verifikasi konsistensi WMO_ID
            data_wmos = set(df['WMO_ID'].unique())
            if len(data_wmos) > 1:
                raise ValueError(f"File mengandung {len(data_wmos)} WMO_ID: {data_wmos}")
            if wmo_id not in data_wmos:
                raise ValueError(f"Ketidaksesuaian WMO_ID: filename={wmo_id}, data={data_wmos}")
            
            # Tambahkan metadata
            filename = os.path.basename(f_path)
            df["used_fallback"] = "fallback" in filename
            df["source_file"] = filename
            df["parameter"] = param
            df["baseline"] = baseline
            
            dfs.append(df)
            
        except Exception as e:
            processing_errors.append((os.path.basename(f_path), str(e)))
            print(f"  ❌ Error proses {os.path.basename(f_path)}: {e}")
            continue
    
    if not dfs:
        print(f"  ❌ Tidak ada data yang berhasil diproses!")
        return False
    
    # Gabungkan semua data valid
    combined = pd.concat(dfs, ignore_index=True)
    print(f"  ✅ Berhasil menggabungkan {len(combined):,} observasi dari {len(dfs)} file")
    
    # 🔒 VALIDASI AKHIR: Pastikan hanya WMO_ID valid yang ada di hasil
    result_wmos = set(combined['WMO_ID'].unique())
    unexpected_wmos = result_wmos - valid_wmos
    
    if unexpected_wmos:
        print(f"\n  ⚠️ PERINGATAN: {len(unexpected_wmos)} WMO_ID tidak valid lolos ke hasil akhir!")
        combined = combined[combined['WMO_ID'].isin(valid_wmos)]
        print(f"    → Menghapus {len(unexpected_wmos)} stasiun tidak valid")
    
    # Urutkan dan hapus duplikat
    combined = combined.sort_values(["WMO_ID", "DATA_TIMESTAMP", "baseline"])
    dup_mask = combined.duplicated(subset=["WMO_ID", "DATA_TIMESTAMP", "baseline"], keep=False)
    if dup_mask.any():
        print(f"  ℹ️ Menghapus {dup_mask.sum():,} duplikat observasi...")
        combined = combined.drop_duplicates(
            subset=["WMO_ID", "DATA_TIMESTAMP", "baseline"], 
            keep="last"
        )
    
    # 🔒 ASSERTION FINAL
    final_wmo_count = combined['WMO_ID'].nunique()
    expected_count = len(valid_wmos)
    
    if final_wmo_count != expected_count:
        print(f"\n  ⚠️ Peringatan: Expected {expected_count} stasiun, got {final_wmo_count}")
        missing = valid_wmos - result_wmos
        if missing:
            print(f"    Stasiun valid yang hilang ({len(missing)}): {sorted(list(missing))[:5]}")
    else:
        print(f"  ✅ Validasi berhasil: {final_wmo_count} stasiun sesuai ekspektasi")
    
    # Simpan hasil akhir
    combined.to_csv(out_csv, index=False)
    print(f"\n  💾 File disimpan: {out_csv}")
    print(f"     Total observasi: {len(combined):,}")
    print(f"     Stasiun unik   : {final_wmo_count}")
    print(f"     Rentang waktu  : {combined['DATA_TIMESTAMP'].min()} s/d {combined['DATA_TIMESTAMP'].max()}")
    
    return True

# ==============================
# 6. EKSEKUSI UTAMA
# ==============================
if __name__ == "__main__":
    print("="*70)
    print("PIPELINE PENGABUNGAN HOMOGENISASI DENGAN DETEKSI KOLOM DINAMIS")
    print("="*70)
    print("\n🔑 Mapping Parameter ke Kolom Nilai:")
    for param, cols in PARAM_TO_VALUE_COL.items():
        print(f"   {param:<25} → {cols}")
    
    summary = {}
    all_success = True
    
    for baseline in BASELINES:
        print(f"\n{'#'*70}")
        print(f"# BASELINE: {baseline}")
        print(f"{'#'*70}")
        
        for param in PARAMS:
            print(f"\n[Parameter: {param}]")
            try:
                success = gabung_homogenisasi(param, baseline)
                key = f"{param}_{baseline}"
                summary[key] = "✅ Berhasil" if success else "❌ Gagal"
                if not success:
                    all_success = False
            except Exception as e:
                print(f"  ❌ Exception tidak terduga: {type(e).__name__}: {e}")
                import traceback
                traceback.print_exc()
                summary[f"{param}_{baseline}"] = f"❌ Error: {e}"
                all_success = False
    
    # Ringkasan akhir
    print(f"\n{'='*70}")
    print("RINGKASAN EKSEKUSI")
    print(f"{'='*70}")
    for key, status in summary.items():
        print(f"{key:<50} : {status}")
    
    if all_success:
        print(f"\n✅ SEMUA PROSES BERHASIL")
        print(f"📁 Output tersedia di: {os.path.abspath(OUTPUT_DIR)}/")
    else:
        print(f"\n⚠️  BEBERAPA PROSES GAGAL - PERIKSA LOG DI ATAS")
        print(f"💡 Tips: Jika masih error kolom, periksa 1 file sampel dengan:")
        print(f"   pd.read_csv('path/ke/file.csv', nrows=0).columns.tolist()")

PIPELINE PENGABUNGAN HOMOGENISASI DENGAN DETEKSI KOLOM DINAMIS

🔑 Mapping Parameter ke Kolom Nilai:
   TEMPERATURE_AVG_C         → ['HOMO_TEMPERATURE_AVG_C', 'temperature', 'temp_avg', 'avg_temp', 'temp', 'value', 'VALUE']
   TEMP_24H_TN_C             → ['HOMO_TEMP_24H_TN_C', 'min_temp', 'temperature_min', 'temp_min', 'value', 'VALUE']
   TEMP_24H_TX_C             → ['HOMO_TEMP_24H_TX_C', 'max_temp', 'temperature_max', 'temp_max', 'value', 'VALUE']

######################################################################
# BASELINE: 1981
######################################################################

[Parameter: TEMPERATURE_AVG_C]

▶ Menggabungkan hasil homogenisasi: TEMPERATURE_AVG_C | Baseline: 1981
  ✅ Terdapat 98 WMO_ID valid (≥80% availability)
  Menemukan 98 file kandidat. Memulai validasi integritas...

  ✅ File VALID   : 98
  ❌ File DITOLAK : 0

  Memproses 98 file valid...
  ✅ Berhasil menggabungkan 1,536,437 observasi dari 98 file
  ✅ Validasi berhasil: 98 stasiun sesua

In [4]:
#############################################################
# GABUNGKAN FILE HOMOGENISASI PER PARAMETER & BASELINE (DENGAN DETEKSI KOLOM DINAMIS)
#############################################################
import pandas as pd
import glob
import os
import re
import shutil
from pathlib import Path

# ==============================
# 1. KONFIGURASI
# ==============================
DATA_DIR = 'data' 
HOMO_DIR   = os.path.join(DATA_DIR, '03.QC_Dataset_Level_02_Homogenisasi')
OUTPUT_DIR = os.path.join(DATA_DIR, '04.Dataset_Final')

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Parameter dan baseline
PARAMS = [
    'TEMPERATURE_AVG_C',
    'TEMP_24H_TN_C',
    'TEMP_24H_TX_C',
]
BASELINES = ['1981', '1991']

# 🔑 MAPPING PARAMETER KE KOLOM NILAI (CRITICAL FIX)
PARAM_TO_VALUE_COL = {
    'TEMPERATURE_AVG_C': ['HOMO_TEMPERATURE_AVG_C','temperature', 'temp_avg', 'avg_temp', 'temp', 'value', 'VALUE'],
    'TEMP_24H_TN_C':     ['HOMO_TEMP_24H_TN_C', 'tn', 'min_temp', 'temperature_min', 'temp_min', 'value', 'VALUE'],
    'TEMP_24H_TX_C':     ['HOMO_TEMP_24H_TX_C', 'tx', 'max_temp', 'temperature_max', 'temp_max', 'value', 'VALUE'],
}

PARAM_TO_TIMESTAMP_COL = ['DATA_TIMESTAMP', 'timestamp', 'time', 'date', 'datetime', 'observation_time']

# ==============================
# 2. VALIDASI KRUSIAL: AMBIL DAFTAR WMO_ID VALID DARI SUMMARY (DINAMIS)
# ==============================
def get_valid_wmo_ids(param, baseline):
    """
    Ambil WMO_ID yang memenuhi kriteria 80% availability.
    Mendukung struktur kolom yang berbeda antar parameter.
    """
    # Mapping parameter ke direktori summary (semua parameter TN/TX menggunakan summary TEMPERATURE_AVG_C)
    summary_path = os.path.join(
        DATA_DIR, 
        '01.QC_Dataset_Level_01', 
        param, 
        '05.Summary', 
        '00.Summary_80percent.csv'
    )
    
    if not os.path.exists(summary_path):
        raise FileNotFoundError(f"File summary tidak ditemukan: {summary_path}")
    
    avail = pd.read_csv(summary_path)
    
    # 🔑 Tentukan kolom ketersediaan dengan strategi fallback dinamis
    col_candidates = [
        f"80%_{param}_{baseline}",
        f"80%_{param}_1981",
        f"80%_TEMPERATURE_AVG_C_{baseline}",
        f"80%_TEMPERATURE_AVG_C_1981",
    ]
    col_name = None
    for candidate in col_candidates:
        if candidate in avail.columns:
            col_name = candidate
            break
    if col_name is None:
        # Cari kolom yang mengandung pola '80%' dan 'TEMP'
        temp_cols = [c for c in avail.columns if '80%' in c and ('TEMP' in c or 'temperature' in c.lower())]
        if temp_cols:
            col_name = temp_cols[0]
        else:
            raise ValueError(
                f"Tidak dapat menemukan kolom ketersediaan 80%. "
                f"Kolom tersedia: {list(avail.columns)}"
            )
    
    # 🔑 Filter stasiun valid
    valid_mask = avail[col_name] == True
    valid_wmos = set(avail[valid_mask]['WMO_ID'].astype(str).unique())
    return valid_wmos

# ==============================
# 3. DETEKSI KOLOM NILAI & TIMESTAMP SECARA DINAMIS
# ==============================
def detect_columns(file_path, param):
    """
    Deteksi kolom nilai dan timestamp yang sesuai dari file CSV.
    Mengembalikan: (value_col, timestamp_col, all_columns)
    """
    # Baca header saja
    df_head = pd.read_csv(file_path, nrows=0)
    cols = [c.lower() for c in df_head.columns]
    original_cols = list(df_head.columns)
    
    # 🔑 Deteksi kolom nilai berdasarkan mapping parameter
    value_candidates = PARAM_TO_VALUE_COL.get(param, ['value', 'VALUE'])
    value_col = None
    
    for candidate in value_candidates:
        if candidate.lower() in cols:
            # Ambil nama kolom original (case-sensitive)
            idx = cols.index(candidate.lower())
            value_col = original_cols[idx]
            break
    
    # 🔑 Deteksi kolom timestamp
    timestamp_col = None
    for candidate in PARAM_TO_TIMESTAMP_COL:
        if candidate.lower() in cols:
            idx = cols.index(candidate.lower())
            timestamp_col = original_cols[idx]
            break
    
    return value_col, timestamp_col, original_cols

# ==============================
# 4. PENGECEKAN INTEGRITAS FILE DENGAN DETEKSI KOLOM
# ==============================
def check_file_integrity(file_path, valid_wmos, param):
    """Validasi komprehensif dengan deteksi kolom dinamis"""
    issues = []
    filename = os.path.basename(file_path)
    
    # 🔒 CEK 1: Pola nama file
    if not re.match(r'^WMO_\d{5,6}_homogen_(fallback|daily)\.csv$', filename):
        issues.append("Pola nama tidak sesuai standar")
    
    # 🔒 CEK 2: Ekstrak WMO_ID dari nama file
    wmo_match = re.search(r'WMO_(\d{5,6})_', filename)
    if not wmo_match:
        issues.append("Tidak dapat ekstrak WMO_ID dari nama file")
        return filename, None, issues, "NAMA_FILE_INVALID"
    
    wmo_id = wmo_match.group(1)
    
    # 🔒 CEK 3: WMO_ID harus ada di daftar valid
    if wmo_id not in valid_wmos:
        issues.append(f"Tidak memenuhi kriteria 80% availability")
        return filename, wmo_id, issues, "TIDAK_MEMENUHI_80PCT"
    
    # 🔒 CEK 4: File harus ada dan tidak kosong
    if not os.path.exists(file_path):
        issues.append("File tidak ditemukan")
        return filename, wmo_id, issues, "FILE_HILANG"
    
    if os.path.getsize(file_path) < 100:
        issues.append("File terlalu kecil (<100 bytes)")
        return filename, wmo_id, issues, "FILE_KOSONG"
    
    # 🔒 CEK 5: Deteksi kolom kritis (nilai & timestamp)
    try:
        value_col, timestamp_col, all_cols = detect_columns(file_path, param)
        
        if value_col is None:
            issues.append(f"Kolom nilai tidak ditemukan. Kolom tersedia: {all_cols}")
            return filename, wmo_id, issues, "KOLOM_NILAI_HILANG"
        
        if timestamp_col is None:
            issues.append(f"Kolom timestamp tidak ditemukan. Kolom tersedia: {all_cols}")
            return filename, wmo_id, issues, "KOLOM_TIMESTAMP_HILANG"
        
        # Simpan info kolom untuk pemrosesan nanti
        return filename, wmo_id, issues, "VALID", value_col, timestamp_col
        
    except Exception as e:
        issues.append(f"Gagal deteksi kolom: {str(e)}")
        return filename, wmo_id, issues, "GAGAL_DETEKSI_KOLOM", None, None
    
    return filename, wmo_id, issues, "VALID", None, None

# ==============================
# 5. FUNGSI GABUNG DENGAN DETEKSI KOLOM DINAMIS
# ==============================
def gabung_homogenisasi(param, baseline):
    print(f"\n▶ Menggabungkan hasil homogenisasi: {param} | Baseline: {baseline}")
    
    in_dir  = os.path.join(HOMO_DIR, f"{param}_BASELINE_{baseline}", 'data')
    out_csv = os.path.join(OUTPUT_DIR, f'{param}_homogen_final_baseline_{baseline}.csv')
    
    # 🔒 VALIDASI 0: Pastikan direktori input ada
    if not os.path.isdir(in_dir):
        print(f"  ❌ Direktori input tidak ditemukan: {in_dir}")
        return False
    
    # 🔒 VALIDASI 1: Ambil daftar WMO_ID valid dari summary
    try:
        valid_wmos = get_valid_wmo_ids(param, baseline)
        print(f"  ✅ Terdapat {len(valid_wmos)} WMO_ID valid (≥80% availability)")
    except Exception as e:
        print(f"  ❌ Gagal validasi WMO_ID: {e}")
        import traceback
        traceback.print_exc()
        return False
    
    # 🔒 VALIDASI 2: Temukan file dengan pola ketat
    pattern = os.path.join(in_dir, "WMO_[0-9]*_homogen_*.csv")
    all_files = sorted(glob.glob(pattern))
    
    if not all_files:
        print(f"  ⚠️ Tidak ada file ditemukan dengan pola: {pattern}")
        return False
    
    print(f"  Menemukan {len(all_files)} file kandidat. Memulai validasi integritas...")
    
    # 🔒 VALIDASI 3: Filter file dengan pengecekan integritas + deteksi kolom
    valid_files = []  # [(file_path, wmo_id, value_col, timestamp_col)]
    rejected_files = []  # [(filename, wmo_id, reason, category)]
    
    for f in all_files:
        result = check_file_integrity(f, valid_wmos, param)
        filename, wmo_id, issues, category = result[0], result[1], result[2], result[3]
        
        if category == "VALID" and not issues:
            # Ekstrak kolom dari hasil deteksi
            value_col, timestamp_col = result[4], result[5]
            valid_files.append((f, wmo_id, value_col, timestamp_col))
        else:
            rejected_files.append((filename, wmo_id, "; ".join(issues), category))
    
    # 🔒 LAPORAN VALIDASI
    print(f"\n  ✅ File VALID   : {len(valid_files)}")
    print(f"  ❌ File DITOLAK : {len(rejected_files)}")
    
    if rejected_files:
        print(f"\n  Detail penolakan (kategori):")
        from collections import Counter
        cat_count = Counter([cat for _,_,_,cat in rejected_files])
        for cat, cnt in cat_count.most_common():
            print(f"    - {cat}: {cnt} file")
        
        # Tampilkan contoh penolakan
        if rejected_files:
            print(f"\n  Contoh file ditolak:")
            for i, (fname, wmo_id, reason, cat) in enumerate(rejected_files[:3], 1):
                wmo_display = wmo_id if wmo_id else "N/A"
                print(f"    {i}. [{cat}] WMO_{wmo_display}: {fname}")
                print(f"        Alasan: {reason[:100]}...")
    
    # 🔒 VALIDASI 4: Pastikan ada file valid untuk diproses
    if not valid_files:
        # 🔑 DIAGNOSTIK KRUSIAL: Periksa struktur kolom 1 file sampel
        if all_files:
            sample_file = all_files[0]
            print(f"\n  🔍 DIAGNOSTIK: Memeriksa struktur kolom file sampel: {os.path.basename(sample_file)}")
            try:
                sample_df = pd.read_csv(sample_file, nrows=0)
                print(f"     Kolom tersedia: {list(sample_df.columns)}")
                print(f"     Mapping yang diharapkan untuk {param}: {PARAM_TO_VALUE_COL.get(param, [])}")
            except Exception as e:
                print(f"     Gagal baca header: {e}")
        
        print(f"\n  ⚠️ Tidak ada file valid untuk diproses!")
        return False
    
    # Proses hanya file valid
    print(f"\n  Memproses {len(valid_files)} file valid...")
    dfs = []
    processing_errors = []
    
    for f_path, wmo_id, value_col, timestamp_col in valid_files:
        try:
            # 🔑 Baca dengan kolom timestamp yang terdeteksi
            parse_dates = [timestamp_col] if timestamp_col else None
            df = pd.read_csv(f_path, parse_dates=parse_dates, low_memory=False)
            
            # 🔑 Normalisasi struktur kolom ke format standar
            # Rename kolom nilai ke 'VALUE' untuk konsistensi
            if value_col != 'VALUE':
                df = df.rename(columns={value_col: 'VALUE'})
            
            # Rename kolom timestamp ke 'DATA_TIMESTAMP' untuk konsistensi
            if timestamp_col and timestamp_col != 'DATA_TIMESTAMP':
                df = df.rename(columns={timestamp_col: 'DATA_TIMESTAMP'})
            
            # Pastikan kolom wajib ada
            required_cols = ['WMO_ID', 'DATA_TIMESTAMP', 'VALUE']
            missing = [c for c in required_cols if c not in df.columns]
            if missing:
                raise ValueError(f"Kolom wajib hilang setelah renaming: {missing}. Kolom tersedia: {list(df.columns)}")
            
            # Normalisasi tipe data
            df['WMO_ID'] = df['WMO_ID'].astype(str)
            df['VALUE'] = pd.to_numeric(df['VALUE'], errors='coerce')
            
            # Verifikasi konsistensi WMO_ID
            data_wmos = set(df['WMO_ID'].unique())
            if len(data_wmos) > 1:
                raise ValueError(f"File mengandung {len(data_wmos)} WMO_ID: {data_wmos}")
            if wmo_id not in data_wmos:
                raise ValueError(f"Ketidaksesuaian WMO_ID: filename={wmo_id}, data={data_wmos}")
            
            # Tambahkan metadata
            filename = os.path.basename(f_path)
            df["used_fallback"] = "fallback" in filename
            df["source_file"] = filename
            df["parameter"] = param
            df["baseline"] = baseline
            
            dfs.append(df)
            
        except Exception as e:
            processing_errors.append((os.path.basename(f_path), str(e)))
            print(f"  ❌ Error proses {os.path.basename(f_path)}: {e}")
            continue
    
    if not dfs:
        print(f"  ❌ Tidak ada data yang berhasil diproses!")
        return False
    
    # Gabungkan semua data valid
    combined = pd.concat(dfs, ignore_index=True)
    print(f"  ✅ Berhasil menggabungkan {len(combined):,} observasi dari {len(dfs)} file")
    
    # 🔒 VALIDASI AKHIR: Pastikan hanya WMO_ID valid yang ada di hasil
    result_wmos = set(combined['WMO_ID'].unique())
    unexpected_wmos = result_wmos - valid_wmos
    
    if unexpected_wmos:
        print(f"\n  ⚠️ PERINGATAN: {len(unexpected_wmos)} WMO_ID tidak valid lolos ke hasil akhir!")
        combined = combined[combined['WMO_ID'].isin(valid_wmos)]
        print(f"    → Menghapus {len(unexpected_wmos)} stasiun tidak valid")
    
    # Urutkan dan hapus duplikat
    combined = combined.sort_values(["WMO_ID", "DATA_TIMESTAMP", "baseline"])
    dup_mask = combined.duplicated(subset=["WMO_ID", "DATA_TIMESTAMP", "baseline"], keep=False)
    if dup_mask.any():
        print(f"  ℹ️ Menghapus {dup_mask.sum():,} duplikat observasi...")
        combined = combined.drop_duplicates(
            subset=["WMO_ID", "DATA_TIMESTAMP", "baseline"], 
            keep="last"
        )

    
    # 🔒 ASSERTION FINAL
    final_wmo_count = combined['WMO_ID'].nunique()
    expected_count = len(valid_wmos)
    
    if final_wmo_count != expected_count:
        print(f"\n  ⚠️ Peringatan: Expected {expected_count} stasiun, got {final_wmo_count}")
        missing = valid_wmos - result_wmos
        if missing:
            print(f"    Stasiun valid yang hilang ({len(missing)}): {sorted(list(missing))[:5]}")
    else:
        print(f"  ✅ Validasi berhasil: {final_wmo_count} stasiun sesuai ekspektasi")
    
    # Simpan hasil akhir
    combined = combined.rename(columns={'VALUE': f'HOMO_{param}'})
    combined.to_csv(out_csv, index=False)
    print(f"\n  💾 File disimpan: {out_csv}")
    print(f"     Total observasi: {len(combined):,}")
    print(f"     Stasiun unik   : {final_wmo_count}")
    print(f"     Rentang waktu  : {combined['DATA_TIMESTAMP'].min()} s/d {combined['DATA_TIMESTAMP'].max()}")
    
    return True

# ==============================
# 6. EKSEKUSI UTAMA
# ==============================
if __name__ == "__main__":
    print("="*70)
    print("PIPELINE PENGABUNGAN HOMOGENISASI DENGAN DETEKSI KOLOM DINAMIS")
    print("="*70)
    print("\n🔑 Mapping Parameter ke Kolom Nilai:")
    for param, cols in PARAM_TO_VALUE_COL.items():
        print(f"   {param:<25} → {cols}")
    
    summary = {}
    all_success = True
    
    for baseline in BASELINES:
        print(f"\n{'#'*70}")
        print(f"# BASELINE: {baseline}")
        print(f"{'#'*70}")
        
        for param in PARAMS:
            print(f"\n[Parameter: {param}]")
            try:
                success = gabung_homogenisasi(param, baseline)
                key = f"{param}_{baseline}"
                summary[key] = "✅ Berhasil" if success else "❌ Gagal"
                if not success:
                    all_success = False
            except Exception as e:
                print(f"  ❌ Exception tidak terduga: {type(e).__name__}: {e}")
                import traceback
                traceback.print_exc()
                summary[f"{param}_{baseline}"] = f"❌ Error: {e}"
                all_success = False
    
    # Ringkasan akhir
    print(f"\n{'='*70}")
    print("RINGKASAN EKSEKUSI")
    print(f"{'='*70}")
    for key, status in summary.items():
        print(f"{key:<50} : {status}")
    
    if all_success:
        print(f"\n✅ SEMUA PROSES BERHASIL")
        print(f"📁 Output tersedia di: {os.path.abspath(OUTPUT_DIR)}/")
    else:
        print(f"\n⚠️  BEBERAPA PROSES GAGAL - PERIKSA LOG DI ATAS")
        print(f"💡 Tips: Jika masih error kolom, periksa 1 file sampel dengan:")
        print(f"   pd.read_csv('path/ke/file.csv', nrows=0).columns.tolist()")

PIPELINE PENGABUNGAN HOMOGENISASI DENGAN DETEKSI KOLOM DINAMIS

🔑 Mapping Parameter ke Kolom Nilai:
   TEMPERATURE_AVG_C         → ['HOMO_TEMPERATURE_AVG_C', 'temperature', 'temp_avg', 'avg_temp', 'temp', 'value', 'VALUE']
   TEMP_24H_TN_C             → ['HOMO_TEMP_24H_TN_C', 'tn', 'min_temp', 'temperature_min', 'temp_min', 'value', 'VALUE']
   TEMP_24H_TX_C             → ['HOMO_TEMP_24H_TX_C', 'tx', 'max_temp', 'temperature_max', 'temp_max', 'value', 'VALUE']

######################################################################
# BASELINE: 1981
######################################################################

[Parameter: TEMPERATURE_AVG_C]

▶ Menggabungkan hasil homogenisasi: TEMPERATURE_AVG_C | Baseline: 1981
  ✅ Terdapat 98 WMO_ID valid (≥80% availability)
  Menemukan 98 file kandidat. Memulai validasi integritas...

  ✅ File VALID   : 98
  ❌ File DITOLAK : 0

  Memproses 98 file valid...
  ✅ Berhasil menggabungkan 1,536,437 observasi dari 98 file
  ✅ Validasi berhasil: 98 s